# 03 · Embeddingi: od tekstu do wektorów i mapy 3D

Embedding to wektor liczb uczony do określonego zadania modelu. Tu osadzamy sekcje sześciu dokumentów Acme w **tej samej tekstowej przestrzeni**. Wektor nie zawiera prostych pól „kwota” lub „data”; jego współrzędne nie są ludzkimi etykietami. Podobieństwo jest przydatne do wyszukania kandydatów, ale nie oznacza prawdy, uprawnień ani poprawnej odpowiedzi.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import base64
import os
import sys
from io import BytesIO
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import numpy as np
import umap
from dotenv import dotenv_values
from PIL import Image as PILImage, ImageDraw
from sklearn.cluster import KMeans

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import DEMO_IDS, embed_texts, ollama_settings, section_records

## 1. Co dokładnie osadzamy?

Nie przekazujemy modelowi metadanych dostępu jako części semantycznego tekstu. Zachowujemy je obok wektora. Każdy rekord wskazuje wersję i hash konkretnej sekcji; zmiana treści wymaga ponownego embeddingu.

In [ ]:
records = section_records()
host, model_name = ollama_settings()
vectors = embed_texts([row["text"] for row in records])
{"documents": len(DEMO_IDS), "sections": len(records), "shape": vectors.shape, "model": model_name,
 "first_record": {key: records[0][key] for key in ("id", "heading", "content_hash")}}

## 2. Kierunek i długość wektora

Cosinus porównuje kierunki: `dot(a,b) / (||a||·||b||)`. Nie porównuj wektorów z różnych modeli lub wymiarów. Normalizacja do długości 1 pozwala użyć iloczynu skalarnego, ale nie naprawia błędnych lub urwanych chunków.

In [ ]:
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
finance = next(i for i, row in enumerate(records) if row["document_id"] == "FIN-YTD-2011" and "Monthly" in row["heading"])
forecast = next(i for i, row in enumerate(records) if row["document_id"] == "SCI-FORECAST-2011" and "Error" in row["heading"])
{"finance_heading": records[finance]["heading"], "forecast_heading": records[forecast]["heading"],
 "cosine": float(unit[finance] @ unit[forecast]), "unit_norm": float(np.linalg.norm(unit[finance]))}

## 3. UMAP 3D i klastrowanie

Najpierw klastrujemy **oryginalne wektory** (eksperyment: dwa zbiory), potem rysujemy ich projekcję UMAP. Kolor dokumentu to znana etykieta źródła, kolor klastra to wynik algorytmu. UMAP zniekształca odległości i gęstość; położenie w 3D nie jest „prawdziwą mapą znaczenia”, a KMeans wymusza liczbę grup. Użyj wykresu do diagnozy, nie do deklarowania jakości retrievalu.

In [ ]:
clusters = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(unit)
coords = umap.UMAP(n_components=3, n_neighbors=min(10, len(records)-1), metric="cosine", random_state=42).fit_transform(unit)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
colors = ["#2358a8" if row["document_id"].startswith("FIN") else "#dc7236" for row in records]
ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=colors, marker="o", alpha=.75)
ax.set(title="Sekcje Acme: niebieski = FIN, pomarańczowy = SCI", xlabel="UMAP 1", ylabel="UMAP 2", zlabel="UMAP 3")
plt.show()
{"cluster_sizes": np.bincount(clusters).tolist(), "finance_in_cluster_0": sum(row["document_id"].startswith("FIN") and label == 0 for row, label in zip(records, clusters))}

## 4. Tekst, tabela, obraz: trzy różne reprezentacje

- Tekst: osadź konkretną sekcję lub pytanie tym samym modelem.
- Tabela: osadź opis z nagłówkami kolumn, zachowując osobno liczby i źródłowy wiersz; do rachunków wróć do danych strukturalnych.
- Obraz: model tekstowy może osadzić **podpis** z lekcji 02, ale nie piksele. Wspólna przestrzeń obraz–tekst wymaga innego, specjalnie wytrenowanego modelu multimodalnego i zgodnej funkcji `embed` dla obu modalności.

Poniżej budujemy mały obraz bez pliku źródłowego z poprzedniej lekcji. `embeddinggemma-2` osadza piksele i oba teksty w **jednej własnej** przestrzeni 768D. Nie porównujemy tego wyniku z 1024D wektorami Qwen powyżej.

In [ ]:
image = PILImage.new("RGB", (500, 110), "white")
ImageDraw.Draw(image).text((18, 45), "FORECAST  ->  ACTUAL  ->  ERROR", fill="black")
buffer = BytesIO()
image.save(buffer, format="PNG")
multimodal_model = os.environ.get("RAG_MULTIMODAL_EMBED_MODEL") or dotenv_values(ROOT / ".env").get("RAG_MULTIMODAL_EMBED_MODEL") or "embeddinggemma-2:latest"
mixed_input = [{"image": base64.b64encode(buffer.getvalue()).decode("ascii")},
               {"text": "A diagram comparing forecast, actual demand and error"},
               {"text": "A report on cash receipts and allocations"}]
response = httpx.post(host.rstrip("/") + "/api/embed", json={"model": multimodal_model, "input": mixed_input, "truncate": False}, timeout=180)
response.raise_for_status()
shared = np.asarray(response.json()["embeddings"], dtype=np.float32)
shared /= np.linalg.norm(shared, axis=1, keepdims=True)
{"shape": shared.shape, "image_to_forecast_text": float(shared[0] @ shared[1]),
 "image_to_cash_text": float(shared[0] @ shared[2])}

**Uwaga:** dwa cosinusy dla pojedynczego rysunku nie mierzą jakości modelu. Potrzebujesz zestawu podpisanych obrazów i pytań, by ocenić retrieval między modalnościami.

**Ćwiczenie:** zmień `n_neighbors` oraz liczbę klastrów. Czy finansowe i naukowe fragmenty stale są rozdzielone? Sprawdź to na oryginalnych tekstach, nie tylko na kolorach wykresu.